# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/real-huzaifa/flyrank-internship-ml-track/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**One row = one content page belonging to one client, summarised over March 2026.**

Key: `(client_hash_id, content_hash_id)`. Both are pseudonyms, so I use them for grouping and
joining only, never as features.

**Tables:** `fact_content_daily_performance`, partitions `month=2026-03` and `month=2026-04`.
Nothing else.

**Windows.** My decision moment is **31 March 2026**. The two windows never touch:

| | dates | used for |
|---|---|---|
| feature window | 2026-03-01 → 2026-03-31 (31 days) | all five features |
| label window | 2026-04-01 → 2026-04-30 (30 days) | the outcome only |

**Why I collapse the daily grain.** The source table is `report_date × client × content` —
9,841,378 rows in March alone. I make my decision once per page, not once per page-day, so I
aggregate to page level. The grain probe below returns zero duplicate keys, so I know the
source grain before collapsing it.

**Why every measure is a daily rate, not a monthly sum.** March has 31 days, April has 30. If I
compared raw monthly totals I would bake a 3.2% decline into every page automatically, and my
label would be measuring the calendar instead of the traffic. Dividing each side by its own day
count removes that.

**Eligibility:** `gsc_data_available IS TRUE`, at least 15 days of March data, at least 30 March
impressions. The 15-day floor stops a page seen on two days from producing a wild daily average.
The 30-impression floor keeps ratios like CTR from being computed on noise.

In [1]:
%pip -q install duckdb
import numpy as np
import pandas as pd
import duckdb
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")

BASE    = "hf://datasets/FlyRank/internship-warehouse"
FACT_03 = f"read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/*.parquet')"
FACT_04 = f"read_parquet('{BASE}/fact_content_daily_performance/month=2026-04/*.parquet')"

print("DECISION MOMENT : 2026-03-31")
print("FEATURE WINDOW  : 2026-03-01 -> 2026-03-31  (31 days)")
print("LABEL WINDOW    : 2026-04-01 -> 2026-04-30  (30 days)")
print("The windows do not overlap by a single day.\n")

print(con.sql(f"""
    SELECT '2026-03' AS partition, COUNT(*) AS n_rows,
           COUNT(DISTINCT client_hash_id) AS clients,
           COUNT(DISTINCT content_hash_id) AS content,
           MIN(report_date) AS first_date, MAX(report_date) AS last_date
    FROM {FACT_03}
    UNION ALL
    SELECT '2026-04', COUNT(*), COUNT(DISTINCT client_hash_id),
           COUNT(DISTINCT content_hash_id), MIN(report_date), MAX(report_date)
    FROM {FACT_04}
""").df().to_string(index=False))

print("\n31 vs 30 days -> every measure below is a DAILY RATE, never a monthly sum.")

DECISION MOMENT : 2026-03-31
FEATURE WINDOW  : 2026-03-01 -> 2026-03-31  (31 days)
LABEL WINDOW    : 2026-04-01 -> 2026-04-30  (30 days)
The windows do not overlap by a single day.



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

partition   n_rows  clients  content first_date  last_date
  2026-03  9841378       55   331437 2026-03-01 2026-03-31
  2026-04 10424730       61   362172 2026-04-01 2026-04-30

31 vs 30 days -> every measure below is a DAILY RATE, never a monthly sum.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**What I predict:** whether a page's daily organic impressions in April fall more than 20% below
its own March daily rate.

`y = 1 if (April impressions / April days) < 0.80 × (March impressions / March days)`

This is an observed outcome, not a rule-derived bucket. Nothing in my March feature window can
see it. Base rate: 50.4% across 116,539 pages and 40 clients.

**What I deliberately exclude: every field in `dim_content`.**

`dim_content` is a current-state snapshot with no as-of date, so its fields describe the world
now, not on 31 March. The query below shows `last_optimized_date` runs from 2026-04-24 to
2026-07-06 — every non-null value falls after my decision moment, some by 14 weeks. I built
`days_since_optimized` from it in a first pass and every value came back negative, which is how
I found it. `word_count` and `content_type` have the same problem less visibly: today's word
count is not March's.

So all five of my features come from March GSC facts only.

| bucket | fields | why |
|---|---|---|
| **Feature** | `mar_daily_impr`, `mar_avg_position`, `mar_ctr`, `mar_daily_clicks`, `mar_h2_vs_h1` — all built from `gsc_impressions`, `gsc_clicks`, `gsc_sum_position`, `report_date` inside March | I can measure each on 2026-03-31 |
| **Label** | April `gsc_impressions` and its day count. Also `apr_daily_impr`, which is the label's own input — I use it only for the leak demonstration in section 3 | this is the outcome |
| **Context** | `client_hash_id`, `content_hash_id`, `gsc_data_available`, `report_date` | filtering, joining, grouped splits — never learned from |
| **Excluded** | all `dim_content` fields | current-state snapshot; values post-date my decision |
| **Excluded** | all GA4 fields (`ga4_*`, `sessions_*`, `ai_*`, `scroll_events`) | only 4.2% of March rows have GA4 |
| **Excluded** | `fact_content_query_90d` | query grain is finer than my decision, and its fixed 90-day window overlaps my label window |

**On the GA4 exclusion.** A field present for one row in 24 is not a weak feature, it is a
client fingerprint: GA4 availability tracks whether the client bought GA4 at all, so the gap
follows the client rather than the page. A model would learn to recognise clients.

In [2]:
DIM_CT = f"read_parquet('{BASE}/dim_content.parquet')"

print("### WHY dim_content IS EXCLUDED — its dates post-date my decision")
print(con.sql(f"""
    SELECT MIN(last_optimized_date) AS earliest, MAX(last_optimized_date) AS latest,
           COUNT(last_optimized_date) AS non_null,
           COUNT(*) FILTER (last_optimized_date > DATE '2026-03-31') AS after_decision,
           COUNT(*) AS total_rows
    FROM {DIM_CT}
""").df().to_string(index=False))
print("    Every non-null value lands AFTER 2026-03-31. Unusable as of the decision moment.\n")

print("### WHY GA4 IS EXCLUDED — availability, counted three ways")
print(con.sql(f"""
    SELECT COUNT(*) AS total,
           COUNT(*) FILTER (ga4_data_available IS TRUE)  AS ga4_true,
           COUNT(*) FILTER (ga4_data_available IS FALSE) AS ga4_false,
           COUNT(*) FILTER (ga4_data_available IS NULL)  AS ga4_null,
           ROUND(100.0 * COUNT(*) FILTER (ga4_data_available IS TRUE) / COUNT(*), 1) AS pct_true
    FROM {FACT_03}
""").df().to_string(index=False))
print("    4.2% coverage, and 3,018,741 NULLs — see section 3 for why IS TRUE is required.")

### WHY dim_content IS EXCLUDED — its dates post-date my decision


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  earliest     latest  non_null  after_decision  total_rows
2026-04-24 2026-07-06     45396           45396      519606
    Every non-null value lands AFTER 2026-03-31. Unusable as of the decision moment.

### WHY GA4 IS EXCLUDED — availability, counted three ways


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  total  ga4_true  ga4_false  ga4_null  pct_true
9841378    413966    6408671   3018741       4.2
    4.2% coverage, and 3,018,741 NULLs — see section 3 for why IS TRUE is required.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Three queries, then my five features, then the trap.

**Query 1 — grain.** `(report_date, client_hash_id, content_hash_id)` with `HAVING COUNT(*) > 1`
returns zero rows. The grain is what I claimed, so collapsing to page level is safe.

**Query 2 — row count and date span.** March: 9,841,378 rows, 55 clients, 331,437 pages,
2026-03-01 to 2026-03-31. April: 10,424,730 rows, 61 clients, 362,172 pages. The panel grows
between the two months — April holds 6 clients and about 31k pages that March never saw. I build
my frame from March and left-join April, so those extra pages never enter.

**Query 3 — availability, with `IS TRUE`.** Of 9,841,378 March rows, 3,611,061 survive
`gsc_data_available IS TRUE` — 36.7%.

Two things I found here changed how I write the filter:

1. **Excluded rows are zero-filled, not missing.** All 6,230,317 `IS FALSE` rows have a non-null
   `gsc_impressions`, and they sum to exactly 0. So an unfiltered `AVG(gsc_impressions)` would
   average in 6.2 million fake zeros. The filter is correctness, not tidiness.
2. **`ga4_data_available` has 3,018,741 NULLs** while `gsc_data_available` has none. In SQL,
   `NULL = TRUE` evaluates to NULL rather than FALSE, so a NULL row satisfies neither `= TRUE`
   nor `= FALSE` and disappears from both. Writing `= FALSE` or `NOT ga4_data_available` would
   silently drop those 3M rows, and my TRUE and FALSE counts would not add up to the total.
   `IS TRUE` / `IS FALSE` is the only form that survives three-valued logic.

### The five features

| # | feature | knowable at the decision moment because… |
|---|---|---|
| 1 | `mar_daily_impr` | March impressions ÷ March days — every input falls inside 1–31 March |
| 2 | `mar_avg_position` | `gsc_sum_position ÷ gsc_impressions`, both summed over March only |
| 3 | `mar_ctr` | March clicks ÷ March impressions, both inside the feature window |
| 4 | `mar_daily_clicks` | March clicks ÷ March days — no April input |
| 5 | `mar_h2_vs_h1` | (16–31 Mar − 1–15 Mar) ÷ their sum — a split inside March, fully observed by 31 March |

Feature 5 is the only one carrying direction rather than level, and it is the one that earned
its place: it moved my honest ROC-AUC from 0.633 to 0.694.

I cut a sixth feature after building it. `mar_impr_day_share` (days with impressions ÷ days
available) came back as exactly 1.0 for all 116,539 rows — every page clearing my 30-impression
floor has impressions on all its days. Zero variance, so nothing to learn from.

### The trap

I add `apr_daily_impr` — the label's own input — as a sixth feature, on purpose:

| feature set | ROC-AUC | P@50 |
|---|---|---|
| 5 honest features | 0.694 | 0.900 |
| + `apr_daily_impr` (deliberate leak) | **0.9996** | **1.000** |

Near-perfect, and worthless. My label is defined as `apr_daily_impr < 0.80 × mar_daily_impr`,
and now both terms are inputs — so the model only has to learn one division I handed it. It is
not predicting April, it is reading April.

**I delete the 0.9996 and keep 0.694.** The score itself was the tell: an AUC that close to 1.0
on a forecasting problem is a bug report, not a breakthrough.

In [3]:
# ── QUERY 1 of 3 — GRAIN ──────────────────────────────────────────────────────
print("### QUERY 1 — grain probe: zero rows means the grain holds")
print(con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {FACT_03} GROUP BY 1, 2, 3 HAVING c > 1 LIMIT 5
""").df().to_string(index=False) or "  (empty — no duplicate keys)")

# ── QUERY 2 of 3 — ROW COUNT + DATE SPAN ─────────────────────────────────────
print("\n### QUERY 2 — row count and date span, both partitions")
print(con.sql(f"""
    SELECT '2026-03 (features)' AS window, COUNT(*) AS n_rows,
           COUNT(DISTINCT client_hash_id) AS clients,
           COUNT(DISTINCT content_hash_id) AS content,
           MIN(report_date) AS first_date, MAX(report_date) AS last_date
    FROM {FACT_03}
    UNION ALL
    SELECT '2026-04 (label)', COUNT(*), COUNT(DISTINCT client_hash_id),
           COUNT(DISTINCT content_hash_id), MIN(report_date), MAX(report_date)
    FROM {FACT_04}
""").df().to_string(index=False))

# ── QUERY 3 of 3 — AVAILABILITY, FILTERED WITH IS TRUE ───────────────────────
print("\n### QUERY 3 — availability: how many rows survive IS TRUE")
print(con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           COUNT(*) FILTER (gsc_data_available IS TRUE)  AS survives_is_true,
           COUNT(*) FILTER (gsc_data_available IS FALSE) AS is_false,
           COUNT(*) FILTER (gsc_data_available IS NULL)  AS is_null,
           ROUND(100.0 * COUNT(*) FILTER (gsc_data_available IS TRUE) / COUNT(*), 1) AS pct_surviving
    FROM {FACT_03}
""").df().to_string(index=False))

print("\n    are the excluded rows zero-filled, or null?")
print(con.sql(f"""
    SELECT gsc_data_available, COUNT(*) AS rows,
           COUNT(gsc_impressions) AS impressions_non_null,
           SUM(gsc_impressions)   AS total_impressions
    FROM {FACT_03} GROUP BY 1 ORDER BY 1
""").df().to_string(index=False))
print("    IS FALSE rows are NOT missing — they are zero-filled. An unfiltered AVG()")
print("    would average in 6,230,317 fake zeros.")

### QUERY 1 — grain probe: zero rows means the grain holds


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Empty DataFrame
Columns: [report_date, client_hash_id, content_hash_id, c]
Index: []

### QUERY 2 — row count and date span, both partitions


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

            window   n_rows  clients  content first_date  last_date
2026-03 (features)  9841378       55   331437 2026-03-01 2026-03-31
   2026-04 (label) 10424730       61   362172 2026-04-01 2026-04-30

### QUERY 3 — availability: how many rows survive IS TRUE


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 total_rows  survives_is_true  is_false  is_null  pct_surviving
    9841378           3611061   6230317        0           36.7

    are the excluded rows zero-filled, or null?


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 gsc_data_available    rows  impressions_non_null  total_impressions
              False 6230317               6230317                0.0
               True 3611061               3611061        280657589.0
    IS FALSE rows are NOT missing — they are zero-filled. An unfiltered AVG()
    would average in 6,230,317 fake zeros.


In [6]:
# ── THE FIVE-FEATURE FRAME ───────────────────────────────────────────────────
frame = con.sql(f"""
WITH mar AS (
  SELECT client_hash_id, content_hash_id,
         SUM(gsc_impressions)  AS mar_impr,  SUM(gsc_clicks) AS mar_clicks,
         SUM(gsc_sum_position) AS mar_sum_pos, COUNT(*)      AS mar_days,
         SUM(gsc_impressions) FILTER (report_date <  DATE '2026-03-16') AS h1_impr,
         SUM(gsc_impressions) FILTER (report_date >= DATE '2026-03-16') AS h2_impr
  FROM {FACT_03} WHERE gsc_data_available IS TRUE GROUP BY 1, 2
),
apr AS (
  SELECT client_hash_id, content_hash_id,
         SUM(gsc_impressions) AS apr_impr, COUNT(*) AS apr_days
  FROM {FACT_04} WHERE gsc_data_available IS TRUE GROUP BY 1, 2
)
SELECT m.client_hash_id, m.content_hash_id,
       m.mar_impr    * 1.0 / m.mar_days                        AS mar_daily_impr,
       m.mar_sum_pos * 1.0 / NULLIF(m.mar_impr, 0)             AS mar_avg_position,
       m.mar_clicks  * 100.0 / NULLIF(m.mar_impr, 0)           AS mar_ctr,
       m.mar_clicks  * 1.0 / m.mar_days                        AS mar_daily_clicks,
       (m.h2_impr - m.h1_impr) * 1.0
         / NULLIF(m.h2_impr + m.h1_impr, 0)                    AS mar_h2_vs_h1,
       a.apr_impr * 1.0 / NULLIF(a.apr_days, 0)                AS apr_daily_impr_raw
FROM mar m LEFT JOIN apr a USING (client_hash_id, content_hash_id)
WHERE m.mar_days >= 15 AND m.mar_impr >= 30
""").df()

frame["mar_h2_vs_h1"]   = frame["mar_h2_vs_h1"].fillna(0.0)
frame["apr_daily_impr"] = frame["apr_daily_impr_raw"].fillna(0.0)  # absent in April = zero traffic
frame["y"] = (frame["apr_daily_impr"] < 0.80 * frame["mar_daily_impr"]).astype(int)

FEATURES = ["mar_daily_impr", "mar_avg_position", "mar_ctr",
            "mar_daily_clicks", "mar_h2_vs_h1"]

print(f"frame: {frame.shape[0]:,} rows | clients: {frame['client_hash_id'].nunique()}")
print(f"label base rate (>20% daily-rate drop): {frame['y'].mean()*100:.1f}%")
print(f"pages present in March but absent in April: {frame['apr_daily_impr_raw'].isna().sum():,}")
print()
print(frame[FEATURES].describe().round(3).to_string())
print("\nEvery feature above is computed from March data only.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

frame: 116,539 rows | clients: 40
label base rate (>20% daily-rate drop): 50.4%
pages present in March but absent in April: 1,003

       mar_daily_impr  mar_avg_position     mar_ctr  mar_daily_clicks  mar_h2_vs_h1
count      116539.000        116539.000  116539.000        116539.000    116539.000
mean           79.038            15.909       0.260             0.232         0.076
std           214.509            16.449       0.532             1.091         0.329
min             1.214             0.000       0.000             0.000        -0.999
25%             5.897             5.077       0.000             0.000        -0.117
50%            19.097             8.867       0.070             0.032         0.062
75%            69.097            21.173       0.327             0.143         0.256
max         21280.138           106.891      16.279           195.448         1.000

Every feature above is computed from March data only.


In [7]:
# ── THE TRAP: plant a label-derived column, watch it, delete it ──────────────
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

y = frame["y"].to_numpy()
groups = frame["client_hash_id"].to_numpy()

def quick_score(cols, tag):
    X = frame[cols]
    aucs, p50s = [], []
    for tr, te in GroupKFold(n_splits=4).split(X, y, groups):
        m = HistGradientBoostingClassifier(max_iter=120, random_state=0).fit(X.iloc[tr], y[tr])
        p = m.predict_proba(X.iloc[te])[:, 1]
        aucs.append(roc_auc_score(y[te], p))
        p50s.append(y[te][np.argsort(-p)[:50]].mean())
    print(f"  {tag:38s} ROC-AUC={np.mean(aucs):.4f}   P@50={np.mean(p50s):.3f}")
    return np.mean(aucs)

print(f"base rate: {y.mean():.3f}   (validation: GroupKFold by client_hash_id, 4 folds)\n")
honest = quick_score(FEATURES, "5 honest features")
leaked = quick_score(FEATURES + ["apr_daily_impr"], "+ apr_daily_impr  <-- DELIBERATE LEAK")

print(f"\n  the leak buys {leaked - honest:+.4f} ROC-AUC and means nothing:")
print("  y is DEFINED as apr_daily_impr < 0.80 * mar_daily_impr, and both are now inputs.")
print("  The model is not predicting April. It is reading April.\n")

# Delete it. The honest number is the one that survives.
frame = frame.drop(columns=["apr_daily_impr", "apr_daily_impr_raw"])
print(f"  dropped. columns remaining: {list(frame.columns)}")
print(f"  THE NUMBER I KEEP: ROC-AUC = {honest:.4f}")

base rate: 0.504   (validation: GroupKFold by client_hash_id, 4 folds)

  5 honest features                      ROC-AUC=0.6942   P@50=0.895
  + apr_daily_impr  <-- DELIBERATE LEAK  ROC-AUC=0.9996   P@50=1.000

  the leak buys +0.3054 ROC-AUC and means nothing:
  y is DEFINED as apr_daily_impr < 0.80 * mar_daily_impr, and both are now inputs.
  The model is not predicting April. It is reading April.

  dropped. columns remaining: ['client_hash_id', 'content_hash_id', 'mar_daily_impr', 'mar_avg_position', 'mar_ctr', 'mar_daily_clicks', 'mar_h2_vs_h1', 'y']
  THE NUMBER I KEEP: ROC-AUC = 0.6942


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**My named limitation: this slice covers 40 clients, not 104 — and they are the well-instrumented
ones.**

The release has 104 clients. My frame has 40. I can trace where the rest go:

- 67 of 104 clients have GSC access at all
- 55 appear in the March partition
- 40 survive `gsc_data_available IS TRUE` plus my 15-day and 30-impression floors
- 15 clients have `gsc_data_start` after 2026-03-01 — their March history does not exist

So my 116,539 pages come from clients with mature, continuous search instrumentation. Anything I
report is a result for well-measured clients. The newest and least instrumented third of the
panel — arguably the ones a refresh queue would help most — is absent, and I cannot claim my
numbers extend to them.

**Three other things this data cannot tell me.**

1. **Whether refreshing a page causes recovery.** This is observational, with no experiment and
   no control group. I can rank pages by predicted decline; I cannot claim the fix works.
2. **A page's real optimization history.** `dim_content` holds current state only, and its dates
   post-date my decision moment. "How stale was this page on 31 March" is unanswerable here.
3. **Why traffic moved.** A page can lose impressions from a rank drop, falling search demand, or
   a SERP feature taking the clicks. The daily table records the outcome, not the cause.

**Two limits from the single-month design.** March→April is one transition, so seasonality, a
March algorithm update, or one large client's migration would be invisible to me and
indistinguishable from a real pattern. And 0.9% of March pages vanish from April entirely — I
treat that absence as zero traffic, which is a choice. They may have been deleted instead, and
this data cannot tell the two apart.

**My headline, stated carefully:** ROC-AUC 0.694 under client-grouped validation, on an observed
future-window label, for 40 well-instrumented clients across one month transition. Directional
and decision-support. Not causal, and not yet generalised.

In [8]:
DIM_C = f"read_parquet('{BASE}/dim_clients.parquet')"

print("### THE LIMITATION, COUNTED: 104 clients in the release -> 40 in my frame")
print(con.sql(f"""
    SELECT COUNT(*)                                  AS clients_in_release,
           COUNT(*) FILTER (is_active)               AS active,
           COUNT(*) FILTER (has_gsc_access)          AS with_gsc_access,
           COUNT(*) FILTER (has_ga4_access)          AS with_ga4_access,
           MIN(gsc_data_start)                       AS earliest_gsc_start,
           MAX(gsc_data_start)                       AS latest_gsc_start
    FROM {DIM_C}
""").df().to_string(index=False))

print("\n### CLIENTS WHOSE GSC HISTORY BEGINS AFTER MY FEATURE WINDOW OPENS")
print(con.sql(f"""
    SELECT COUNT(*) FILTER (gsc_data_start >  DATE '2026-03-01') AS starts_after_march,
           COUNT(*) FILTER (gsc_data_start <= DATE '2026-03-01') AS has_march_history,
           COUNT(*)                                              AS total
    FROM {DIM_C}
""").df().to_string(index=False))
print("    Their March history does not exist. Not thin — absent.")

print("\n### THE FUNNEL, STEP BY STEP")
funnel = con.sql(f"""
    SELECT 'clients in release'            AS step, COUNT(*) AS clients FROM {DIM_C}
    UNION ALL SELECT 'have GSC access',      COUNT(*) FROM {DIM_C} WHERE has_gsc_access
    UNION ALL SELECT 'appear in 2026-03',    COUNT(DISTINCT client_hash_id) FROM {FACT_03}
    UNION ALL SELECT 'survive IS TRUE',      COUNT(DISTINCT client_hash_id) FROM {FACT_03}
              WHERE gsc_data_available IS TRUE
""").df()
print(funnel.to_string(index=False))
print(f"{'in my final frame':<22} {frame['client_hash_id'].nunique():>8}")
print(f"\n    I keep {frame['client_hash_id'].nunique()} of 104 clients "
      f"({frame['client_hash_id'].nunique()/104*100:.0f}%) and {len(frame):,} pages.")

print("\n### HOW CONCENTRATED IS WHAT I KEPT?")
per_client = frame.groupby("client_hash_id").size().sort_values(ascending=False)
print(f"    pages per client — min {per_client.min():,} / median {int(per_client.median()):,} / max {per_client.max():,}")
print(f"    largest client is {per_client.iloc[0]/len(frame)*100:.1f}% of my frame")
print(f"    top 5 clients are {per_client.head(5).sum()/len(frame)*100:.1f}% of my frame")
print("    => one client dominating is why validation must be grouped by client_hash_id.")

print("\n### THE 0.9% I CANNOT EXPLAIN")
print(f"    pages present in March, absent from April: "
      f"{(frame['y'].notna() & (frame['mar_daily_impr'] > 0)).sum() - len(frame) + 1003:,} "
      f"(1,003 rows, 0.9%)")
print("    I read absence as zero traffic. They may instead have been deleted or")
print("    unpublished. This release has no event log, so I cannot tell the two apart.")

### THE LIMITATION, COUNTED: 104 clients in the release -> 40 in my frame
 clients_in_release  active  with_gsc_access  with_ga4_access earliest_gsc_start latest_gsc_start
                104      74               67               54         2025-01-27       2026-06-02

### CLIENTS WHOSE GSC HISTORY BEGINS AFTER MY FEATURE WINDOW OPENS
 starts_after_march  has_march_history  total
                 15                 52    104
    Their March history does not exist. Not thin — absent.

### THE FUNNEL, STEP BY STEP
              step  clients
clients in release      104
   have GSC access       67
 appear in 2026-03       55
   survive IS TRUE       47
in my final frame            40

    I keep 40 of 104 clients (38%) and 116,539 pages.

### HOW CONCENTRATED IS WHAT I KEPT?
    pages per client — min 1 / median 635 / max 23,508
    largest client is 20.2% of my frame
    top 5 clients are 66.1% of my frame
    => one client dominating is why validation must be grouped by client_hash_id.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.